In [0]:
%run ./config

In [0]:
zepto_schema=StructType(fields=[
    StructField('Date', StringType(), True),
    StructField('SKU Number', StringType(), True),
    StructField('SKU Name', StringType(), True),
    StructField('EAN', LongType(), True),
    StructField('SKU Category', StringType(), True),
    StructField('SKU Sub Category', StringType(), True),
    StructField('Brand Name', StringType(), True),
    StructField('Manufacturer Name', StringType(), True),
    StructField('Manufacturer ID', StringType(), True),
    StructField('City', StringType(), True),
    StructField('Sales (Qty) - Units', IntegerType(), True),
    StructField('MRP', FloatType(), True),
    StructField('Gross Merchandise Value', FloatType(), True)
])

In [0]:
df=spark.read.schema(zepto_schema).option('header', True).csv(bronze+'/zepto.csv')

In [0]:
df= df.withColumn('order_date', try_to_date(col('Date'), 'dd-MM-yyyy'))\
    .drop('Date')\
    .withColumnRenamed('SKU Number', 'sku_id')\
    .withColumn('sku_name', lower(trim('SKU Name')))\
    .withColumnRenamed('EAN', 'ean')\
    .withColumnRenamed('Sales (Qty) - Units', 'total_units')\
    .withColumnRenamed('MRP', 'price')\
    .withColumnRenamed('Gross Merchandise Value', 'total_revenue')\
    .withColumnRenamed('order_date', 'date')


In [0]:
df=df.select("date", 'sku_name',"sku_id" , "total_units", "price", "total_revenue").withColumn('data_source', lit('Zepto'))
df=df.na.drop()

In [0]:
df.write.option('header', True).mode('overwrite').csv(silver+'/zepto.csv')

In [0]:
df=spark.read.option('header', True).option('inferSchema', True).csv(silver+'/zepto.csv')
df=df.select('sku_name').distinct()
df.display()